# Chapter 7 — Softmax + Cross-Entropy (the Loss Head)

> Course: **llm.c — Zero to Hero**, Chapter 7 of ~20.
> Builds on Chapter 6 (you've already met the max-shift softmax inside attention).

After Chapter 6 — easily the heaviest chapter so far — this is going to feel like a vacation. Three short functions, one beautiful identity, and one new concept (`Vp`, the padded vocab size).

The loss head sits at the very top of GPT-2: it takes the logits over the vocabulary, turns them into probabilities, picks out the probability assigned to the correct next token, and reports `-log` of that as the per-position loss. Three operations (softmax → index → log) and you have a number you can backprop through.

The fun is that *all three* compose into a stunningly clean backward formula:

$$\frac{\partial L}{\partial \text{logits}_i} = \text{probs}_i - \mathbb{1}[i = \text{target}]$$

That's it. No chain-rule expansions, no Jacobian-vector products. We'll derive that identity from the chain rule, see why `llm.c` (and basically every Transformer codebase) **fuses** softmax and cross-entropy into a single backward function, and run the result against PyTorch.

### Learning objectives

By the end of this chapter you will:

- Read and write `softmax_forward` (numerically stable max-shift form).
- Read and write `crossentropy_forward` — a 1-line gather plus a `-log`.
- **Derive the fused softmax-cross-entropy gradient** `(probs - one_hot) * dloss` from first principles.
- Explain why fusion saves memory **and** FLOPs **and** numerical stability — the same argument that motivates `fused_classifier.cuh` in Part III.
- Understand `Vp` (padded vocab) vs `V` (true vocab) and why the padding exists.


## 1. Concept — Softmax with the Max-Shift Trick

You already know softmax:

$$\text{softmax}(z)_i = \frac{e^{z_i}}{\sum_j e^{z_j}}$$

In float32, computing `exp(z_i)` directly is dangerous: GPT-2's logits can easily exceed 30, and `expf(30) ≈ 10¹³` is fine, but `expf(89)` overflows to `inf`. Then your sum becomes `inf`, every probability becomes `inf/inf = NaN`, and your loss is `NaN` forever.

The fix is the **max-shift identity**:

$$\text{softmax}(z)_i = \frac{e^{z_i - m}}{\sum_j e^{z_j - m}} \quad \text{for any constant } m$$

Picking $m = \max_j z_j$ makes the largest exponent exactly `0`, so `exp` produces a number in `(0, 1]`. No overflow possible. We saw this exact trick in Chapter 6 inside attention (`maxval = -10000.0f`). Here we apply it again over the vocabulary axis.


## 2. Concept — Cross-Entropy Loss

Given a target token id $y$ at one position and a probability vector $p$ over the vocabulary, the cross-entropy loss is:

$$L = -\log p_y$$

That's it. We grab the probability the model assigned to the *correct* next token, take its negative log, and that's the loss for that position.

The total loss is averaged over all `B*T` positions: $L_\text{total} = \frac{1}{BT}\sum_{b,t} L_{b,t}$. This averaging is why the per-position `dloss` you'll see in the backward is `1/(B*T)` — the gradient of the average with respect to one term.

Why this loss? Because $-\log p_y$ is exactly the negative-log-likelihood of the data under the model. Maximizing data likelihood ↔ minimizing this loss. It's the unique loss that makes the maximum-likelihood interpretation clean.


## 3. Concept — Padded Vocab (`Vp` vs `V`)

GPT-2's vocabulary has `V = 50257` tokens. But the codebase quietly uses a slightly larger number, `Vp = 50304`, in many places. Why?

`50304 = 393 × 128` is the next multiple of 128 above 50257. **Multiple-of-128 sizes give massive performance wins** for GPU matrix multiplies: cuBLAS, tensor cores, and shared-memory tiles are all sized in 32-or-128-element chunks, and a misaligned vocab dimension forces the kernel into a slow fallback or wastes the last partial tile.

The padding strategy:

- **Linear layers and logits** are computed over `Vp` (matmul-friendly).
- **Softmax, cross-entropy, and their backwards** explicitly loop only over `V` — they never *see* the 47 padding entries. Whatever garbage is in `logits[V:Vp]` is ignored, and `probs[V:Vp]` / `dlogits[V:Vp]` are kept at zero.

This split is one of the cleanest examples in the codebase of **letting matmul be matmul** while keeping the math semantically exact. We'll see the `Vp` constant in the model config (Chapter 8) and again on the GPU side.


## 4. PyTorch Baseline

In [ ]:
import torch
import torch.nn.functional as F

torch.manual_seed(0)
B, T, V = 2, 3, 5

logits = torch.randn(B, T, V) * 5.0          # bigger range so the max-shift trick matters
targets = torch.randint(0, V, (B, T))

# numerically-stable softmax + cross-entropy via PyTorch
probs   = F.softmax(logits, dim=-1)          # (B, T, V)
log_p   = F.log_softmax(logits, dim=-1)      # (B, T, V) — the fused log+softmax
losses  = F.nll_loss(log_p.view(B*T, V), targets.view(B*T), reduction='none').view(B, T)
mean_loss = losses.mean()

print("logits[0,0]:", [round(v, 3) for v in logits[0,0].tolist()])
print("probs[0,0]:", [round(v, 4) for v in probs[0,0].tolist()],
      "  sums to", round(probs[0,0].sum().item(), 6))
print("targets[0]:", targets[0].tolist())
print("losses[0]:", [round(v, 4) for v in losses[0].tolist()])
print("mean loss:", round(mean_loss.item(), 4))


Two PyTorch idioms worth knowing:

- `F.log_softmax(x)` is the numerically-stable form of `log(softmax(x))`. It avoids ever computing `softmax(x)` explicitly, dodging both the underflow (very small probs → `log(0) = -inf`) and the overflow we just discussed.
- `F.cross_entropy(logits, targets)` does **softmax + log + NLL** all in one — exactly the fused form we're about to study.


## 5. The C `softmax_forward`

From [`train_gpt2.c`](train_gpt2.c) lines 449–484:

```c
void softmax_forward(float* probs, float* logits, int B, int T, int V, int Vp) {
    // output: probs are (B,T,Vp) of the probabilities (sums to 1.0 in each b,t position)
    // input: logits is (B,T,Vp) of the unnormalized log probabilities
    // example: Vp is 50304 and V is 50257
    #pragma omp parallel for collapse(2)
    for (int b = 0; b < B; b++) {
        for (int t = 0; t < T; t++) {
            float* logits_bt = logits + b*T*Vp + t*Vp;
            float* probs_bt  = probs  + b*T*Vp + t*Vp;

            // pass 1: maxval (numerical stability)
            float maxval = -10000.0f;
            for (int i = 0; i < V; i++)             // <-- only V, ignore padding
                if (logits_bt[i] > maxval) maxval = logits_bt[i];

            // pass 2: exp(x - max), accumulate sum
            float sum = 0.0f;
            for (int i = 0; i < V; i++) {
                probs_bt[i] = expf(logits_bt[i] - maxval);
                sum += probs_bt[i];
            }

            // pass 3: normalize
            for (int i = 0; i < V; i++) probs_bt[i] /= sum;

            // pass 4: zero out the padded slots (defensive, not strictly required)
            for (int i = V; i < Vp; i++) probs_bt[i] = 0.0f;
        }
    }
}
```

The same three-pass softmax we built inside attention, with one new wrinkle: **the strides are `Vp` even though we only iterate over `V`**. `logits_bt + b*T*Vp + t*Vp` skips the right number of elements between rows; the inner loop merely declines to look at the last `Vp - V` of them.

Pass 4 (zeroing the padded slots) isn't strictly necessary — nothing downstream reads them — but it's good hygiene and helps when comparing tensors directly to PyTorch.


## 6. The C `crossentropy_forward`

From [`train_gpt2.c`](train_gpt2.c) lines 486–500:

```c
void crossentropy_forward(float* losses,
                          float* probs, int* targets,
                          int B, int T, int Vp) {
    for (int b = 0; b < B; b++) {
        for (int t = 0; t < T; t++) {
            float* probs_bt = probs + b*T*Vp + t*Vp;
            int ix = targets[b*T + t];
            losses[b*T + t] = -logf(probs_bt[ix]);
        }
    }
}
```

Three lines per `(b, t)`:

1. Pointer to the probability vector at this position.
2. Read the target token id.
3. `-log(probs[target])`.

The output `losses` is a `(B, T)` tensor — one scalar loss per position. The model later averages it.


## 7. Compile and Verify Forward

In [ ]:
!mkdir -p course/ch07_build


In [ ]:
%%writefile course/ch07_build/loss_forward.c
#include <stdio.h>
#include <stdlib.h>
#include <math.h>
#include <omp.h>

void softmax_forward(float* probs, float* logits, int B, int T, int V, int Vp) {
    #pragma omp parallel for collapse(2)
    for (int b = 0; b < B; b++)
        for (int t = 0; t < T; t++) {
            float* logits_bt = logits + b*T*Vp + t*Vp;
            float* probs_bt  = probs  + b*T*Vp + t*Vp;
            float maxval = -10000.0f;
            for (int i = 0; i < V; i++) if (logits_bt[i] > maxval) maxval = logits_bt[i];
            float sum = 0.0f;
            for (int i = 0; i < V; i++) { probs_bt[i] = expf(logits_bt[i] - maxval); sum += probs_bt[i]; }
            for (int i = 0; i < V; i++) probs_bt[i] /= sum;
            for (int i = V; i < Vp; i++) probs_bt[i] = 0.0f;
        }
}

void crossentropy_forward(float* losses, float* probs, int* targets, int B, int T, int Vp) {
    for (int b = 0; b < B; b++)
        for (int t = 0; t < T; t++) {
            float* probs_bt = probs + b*T*Vp + t*Vp;
            int ix = targets[b*T + t];
            losses[b*T + t] = -logf(probs_bt[ix]);
        }
}

static void* rd(const char* p, size_t n) {
    FILE* f = fopen(p, "rb"); if (!f){perror(p); exit(1);}
    void* b = malloc(n); size_t r = fread(b,1,n,f); (void)r; fclose(f); return b;
}

int main(int argc, char** argv) {
    if (argc != 5) { fprintf(stderr, "usage: B T V Vp\n"); return 1; }
    int B=atoi(argv[1]), T=atoi(argv[2]), V=atoi(argv[3]), Vp=atoi(argv[4]);
    float* logits  = (float*) rd("course/ch07_build/logits.bin",  (size_t)B*T*Vp*sizeof(float));
    int*   targets = (int*)   rd("course/ch07_build/targets.bin", (size_t)B*T*sizeof(int));
    float* probs   = (float*) calloc((size_t)B*T*Vp, sizeof(float));
    float* losses  = (float*) malloc((size_t)B*T*sizeof(float));
    softmax_forward(probs, logits, B, T, V, Vp);
    crossentropy_forward(losses, probs, targets, B, T, Vp);
    FILE* f;
    f=fopen("course/ch07_build/probs.bin", "wb"); fwrite(probs, 4,(size_t)B*T*Vp,f); fclose(f);
    f=fopen("course/ch07_build/losses.bin","wb"); fwrite(losses,4,(size_t)B*T,   f); fclose(f);
    free(logits); free(targets); free(probs); free(losses);
    return 0;
}


In [ ]:
!gcc -O3 -Wall -fopenmp -o course/ch07_build/loss_forward course/ch07_build/loss_forward.c -lm


In [ ]:
import numpy as np, torch, torch.nn.functional as F, subprocess

torch.manual_seed(0)
B, T, V = 2, 3, 5
Vp = 8                      # pretend we padded V=5 up to 8 for matmul-friendliness

# Logits over the padded vocab. The first V columns are real; the rest will be ignored.
logits = torch.randn(B, T, Vp) * 5.0
targets = torch.randint(0, V, (B, T))

logits.numpy().astype(np.float32).tofile("course/ch07_build/logits.bin")
targets.numpy().astype(np.int32).tofile("course/ch07_build/targets.bin")

subprocess.run(["./course/ch07_build/loss_forward", str(B), str(T), str(V), str(Vp)], check=True)

probs_c  = np.fromfile("course/ch07_build/probs.bin",  dtype=np.float32).reshape(B, T, Vp)
losses_c = np.fromfile("course/ch07_build/losses.bin", dtype=np.float32).reshape(B, T)

# PyTorch reference, using only the real V columns
probs_pt  = F.softmax(logits[:, :, :V], dim=-1).numpy()
losses_pt = F.cross_entropy(logits[:, :, :V].reshape(B*T, V), targets.view(B*T), reduction='none').view(B, T).numpy()

print(f"probs (real V) diff: {np.max(np.abs(probs_c[:, :, :V] - probs_pt)):.2e}")
print(f"losses diff:         {np.max(np.abs(losses_c - losses_pt)):.2e}")
print(f"probs row sums (should be 1.0):", probs_c[:, :, :V].sum(axis=-1).flatten().round(6))
print(f"padded slots zero:", np.all(probs_c[:, :, V:] == 0.0))


## 8. The Fused Backward — Math

Here's where it gets pretty. We want $\partial L / \partial \text{logits}_i$ at one position. Decompose via chain rule through the two operations:

**Step 1: cross-entropy → probs**

$$L = -\log p_y \quad \Rightarrow \quad \frac{\partial L}{\partial p_i} = \begin{cases} -1/p_y & \text{if } i = y \\ 0 & \text{otherwise} \end{cases}$$

So `dprobs` has only **one nonzero entry** — at the target index.

**Step 2: softmax Jacobian**

For $p_k = \text{softmax}(z)_k$, we know

$$\frac{\partial p_k}{\partial z_i} = p_k\,(\delta_{k,i} - p_i)$$

**Step 3: chain them**

$$\frac{\partial L}{\partial z_i} = \sum_k \frac{\partial L}{\partial p_k}\,\frac{\partial p_k}{\partial z_i}$$

The sum collapses because $\partial L / \partial p_k = 0$ except at $k = y$:

$$\frac{\partial L}{\partial z_i} = \frac{\partial L}{\partial p_y}\,\frac{\partial p_y}{\partial z_i} = \Big(-\frac{1}{p_y}\Big)\,p_y\,(\delta_{y,i} - p_i)$$

The $1/p_y$ and $p_y$ cancel, and we land at:

$$\boxed{\frac{\partial L}{\partial z_i} = p_i - \delta_{y,i}}$$

**That's it.** The gradient of the cross-entropy with respect to the logit `i` is simply the probability `i` minus the indicator that `i` is the target. No `1/p_y` divide, no Jacobian matrix, no log-sum-exp manipulation.

If we further account for the loss being averaged over `B*T` positions (`dloss = 1/(B*T)` flowing in from the mean), we get:

$$\frac{\partial L_\text{total}}{\partial z_{b,t,i}} = (p_{b,t,i} - \delta_{y_{b,t}, i}) \cdot \text{dloss}_{b,t}$$

which is precisely what the C code below computes.


## 9. The C `crossentropy_softmax_backward`

From [`train_gpt2.c`](train_gpt2.c) lines 502–521:

```c
void crossentropy_softmax_backward(float* dlogits,
                           float* dlosses, float* probs, int* targets,
                           int B, int T, int V, int Vp) {
    for (int b = 0; b < B; b++) {
        for (int t = 0; t < T; t++) {
            float* dlogits_bt = dlogits + b*T*Vp + t*Vp;
            float* probs_bt   = probs   + b*T*Vp + t*Vp;
            float dloss = dlosses[b*T + t];
            int ix = targets[b*T + t];
            // note we only loop to V, leaving the padded dimensions of dlogits at zero
            for (int i = 0; i < V; i++) {
                float p = probs_bt[i];
                float indicator = (i == ix) ? 1.0f : 0.0f;
                dlogits_bt[i] += (p - indicator) * dloss;
            }
        }
    }
}
```

**One inner loop. Three terms in the body. No softmax Jacobian explicitly built.** This is the entire reason fused softmax+CE is the standard pattern in every Transformer codebase.

The function name is the giveaway: `crossentropy_softmax_backward` — *one* function backprops through *both* operations.


## 10. Why Fuse? — Memory, FLOPs, Stability

Let's count what fusion saves at GPT-2 small scale (`B=4, T=1024, V=50257`):

### Memory

A non-fused implementation would need to materialize the intermediate `dprobs` tensor of shape `(B, T, V)` — about **820 MB** in float32 — purely to throw it away after one matmul. Fused: zero extra memory.

### FLOPs

A naive softmax-Jacobian-then-multiply costs $O(V^2)$ per position — ~2.5 billion FLOPs *per position*, ~10 trillion FLOPs total per backward step. Fused: $O(V)$ per position, ~50 thousand FLOPs per position. **About 50,000× fewer operations.** This is not a small win.

### Numerical stability

The naive form involves explicit `1/probs[target]` division. If the model has just been initialized, `probs[target]` may be `1/V ≈ 2e-5` and that's already on the edge of float32 precision. The fused form **never divides by a small probability**.

### Why this matters in real life

In production GPU kernels (`fused_classifier.cuh`, Chapter 15) we go even further: the *forward* loss computation and the *backward* gradient are produced in the same kernel pass over the logits. The forward computes `loss = -log(probs[target])` and the backward computes `dlogits = (probs - one_hot) * dloss`, both from the same softmax probabilities — saving a second pass over a 50k-wide vector per position.


## 11. Compile and Verify Backward

In [ ]:
%%writefile course/ch07_build/loss_backward.c
#include <stdio.h>
#include <stdlib.h>

void crossentropy_softmax_backward(float* dlogits, float* dlosses, float* probs, int* targets,
                                   int B, int T, int V, int Vp) {
    for (int b = 0; b < B; b++)
        for (int t = 0; t < T; t++) {
            float* dlogits_bt = dlogits + b*T*Vp + t*Vp;
            float* probs_bt   = probs   + b*T*Vp + t*Vp;
            float dloss = dlosses[b*T + t];
            int ix = targets[b*T + t];
            for (int i = 0; i < V; i++) {
                float p = probs_bt[i];
                float indicator = (i == ix) ? 1.0f : 0.0f;
                dlogits_bt[i] += (p - indicator) * dloss;
            }
        }
}

static void* rd(const char* p, size_t n) {
    FILE* f = fopen(p, "rb"); if (!f){perror(p); exit(1);}
    void* b = malloc(n); size_t r = fread(b,1,n,f); (void)r; fclose(f); return b;
}

int main(int argc, char** argv) {
    if (argc != 5) return 1;
    int B=atoi(argv[1]), T=atoi(argv[2]), V=atoi(argv[3]), Vp=atoi(argv[4]);
    float* probs   = (float*) rd("course/ch07_build/probs.bin",   (size_t)B*T*Vp*sizeof(float));
    float* dlosses = (float*) rd("course/ch07_build/dlosses.bin", (size_t)B*T*sizeof(float));
    int*   targets = (int*)   rd("course/ch07_build/targets.bin", (size_t)B*T*sizeof(int));
    float* dlogits = (float*) calloc((size_t)B*T*Vp, sizeof(float));
    crossentropy_softmax_backward(dlogits, dlosses, probs, targets, B, T, V, Vp);
    FILE* f = fopen("course/ch07_build/dlogits.bin","wb"); fwrite(dlogits,4,(size_t)B*T*Vp,f); fclose(f);
    free(probs); free(dlosses); free(targets); free(dlogits);
    return 0;
}


In [ ]:
!gcc -O3 -Wall -o course/ch07_build/loss_backward course/ch07_build/loss_backward.c


In [ ]:
# Run autograd on logits[:, :, :V]; compare dlogits to (probs - onehot) / (B*T)
import numpy as np, torch, torch.nn.functional as F, subprocess

torch.manual_seed(0)
B, T, V = 2, 3, 5
Vp = 8

logits  = (torch.randn(B, T, Vp) * 5.0).requires_grad_()
targets = torch.randint(0, V, (B, T))

# Loss is mean over B*T (so dloss = 1/(B*T) per position)
loss = F.cross_entropy(logits[:, :, :V].reshape(B*T, V), targets.view(B*T), reduction='mean')
loss.backward()

# Save inputs for the C backward
probs_pt = F.softmax(logits.detach()[:, :, :V], dim=-1)
# Match the C side: probs over Vp, with the padded slots zero
probs_full = torch.zeros(B, T, Vp); probs_full[:, :, :V] = probs_pt
probs_full.numpy().astype(np.float32).tofile("course/ch07_build/probs.bin")
targets.numpy().astype(np.int32).tofile("course/ch07_build/targets.bin")
dlosses_per = torch.full((B, T), 1.0/(B*T), dtype=torch.float32)   # uniform 1/(B*T)
dlosses_per.numpy().tofile("course/ch07_build/dlosses.bin")

subprocess.run(["./course/ch07_build/loss_backward", str(B), str(T), str(V), str(Vp)], check=True)
dlogits_c = np.fromfile("course/ch07_build/dlogits.bin", dtype=np.float32).reshape(B, T, Vp)

# Autograd's dlogits is over the full Vp; only the first V columns are non-zero
print(f"dlogits[:, :, :V] diff: {np.max(np.abs(dlogits_c[:, :, :V] - logits.grad[:, :, :V].numpy())):.2e}")
print(f"dlogits[:, :, V:] are all zero in C: {np.all(dlogits_c[:, :, V:] == 0.0)}")


## 12. Toy — `(probs - one_hot)` Made Concrete

A clean way to internalize the fused gradient: imagine the model's predicted distribution `probs` and the target as a one-hot `δ_y`. The gradient is **literally the difference**.

Three regimes worth seeing on paper:

1. **Confident-and-correct**: `probs[target] ≈ 1`, others ≈ 0. Then `(probs - one_hot) ≈ 0` everywhere. **Almost no gradient flows.** The model is happy with this prediction; nothing to push.
2. **Confident-and-wrong**: `probs[wrong_class] ≈ 1`, `probs[target] ≈ 0`. Then `(probs - one_hot)` has `+1` at the wrong class and `-1` at the target. **Big push** to lower the wrong probability and raise the target.
3. **Uniform / untrained**: `probs[i] ≈ 1/V`. The gradient is `~1/V` at non-target indices and `~1/V - 1` at the target. Symmetric soft push everywhere.

Run a tiny example.


In [ ]:
%%writefile course/ch07_build/toy_grad.c
#include <stdio.h>

int main(void) {
    const int V = 4;
    float dloss = 1.0f;   // pretend B*T = 1, so dloss = 1
    int target = 2;

    float scenarios[3][4] = {
        {0.01f, 0.02f, 0.95f, 0.02f},  // confident & correct
        {0.45f, 0.45f, 0.05f, 0.05f},  // confident & wrong (mass on 0/1)
        {0.25f, 0.25f, 0.25f, 0.25f},  // uniform
    };
    const char* labels[] = {"confident & correct", "confident & wrong  ", "uniform            "};

    for (int s = 0; s < 3; s++) {
        printf("%s probs = [%.2f %.2f %.2f %.2f] target=%d\n",
               labels[s], scenarios[s][0], scenarios[s][1], scenarios[s][2], scenarios[s][3], target);
        printf("                       dlogits = [");
        for (int i = 0; i < V; i++) {
            float indicator = (i == target) ? 1.0f : 0.0f;
            float g = (scenarios[s][i] - indicator) * dloss;
            printf("%+.3f ", g);
        }
        printf("]\n\n");
    }
    return 0;
}


In [ ]:
!gcc -O2 -Wall -o course/ch07_build/toy_grad course/ch07_build/toy_grad.c && ./course/ch07_build/toy_grad


You can read the magnitude of the gradient as **how much the model is pushed away from its current prediction toward the target**. Note that all three rows sum to zero — that's because $\sum_i (p_i - \delta_{y,i}) = 1 - 1 = 0$, a consequence of probabilities summing to 1. The gradient lives on the simplex's tangent space.


## 13. TODO Exercise 1 — Write `softmax_forward`

Boilerplate provided. Fill in the four passes (max, exp+sum, normalize, zero padded slots).


In [ ]:
%%writefile course/ch07_build/exercise1.c
#include <stdio.h>
#include <stdlib.h>
#include <math.h>
#include <omp.h>

void softmax_forward(float* probs, float* logits, int B, int T, int V, int Vp) {
    #pragma omp parallel for collapse(2)
    for (int b = 0; b < B; b++)
        for (int t = 0; t < T; t++) {
            float* logits_bt = logits + b*T*Vp + t*Vp;
            float* probs_bt  = probs  + b*T*Vp + t*Vp;

            // TODO 1: find maxval = max(logits_bt[0..V))
            float maxval = -10000.0f;
            for (int i=0; i<V; i++)
                if (logits_bt[i] > maxval) maxval = logits_bt[i];

            // TODO 2: exp(x - maxval), accumulate sum
            float sum = 0.0f;
            for (int i=0; i<V; i++) {
                probs_bt[i] = expf(logits_bt[i] - maxval);
                sum += probs_bt[i];
            }

            // TODO 3: normalize probs_bt[0..V) by dividing by sum
            for (int i=0; i<V; i++) probs_bt[i] /= sum;

            // TODO 4: zero out probs_bt[V..Vp)
            (void)maxval; (void)sum;
            for (int i=V; i<Vp; i++) probs_bt[i] = 0.0f;
        }
}

static void* rd(const char* p, size_t n){FILE*f=fopen(p,"rb");void*b=malloc(n);size_t r=fread(b,1,n,f);(void)r;fclose(f);return b;}

int main(int argc, char** argv) {
    int B=atoi(argv[1]), T=atoi(argv[2]), V=atoi(argv[3]), Vp=atoi(argv[4]);
    float* logits = (float*) rd("course/ch07_build/logits.bin", (size_t)B*T*Vp*sizeof(float));
    float* probs  = (float*) calloc((size_t)B*T*Vp, sizeof(float));
    softmax_forward(probs, logits, B, T, V, Vp);
    FILE* f=fopen("course/ch07_build/probs_ex1.bin","wb"); fwrite(probs,4,(size_t)B*T*Vp,f); fclose(f);
    free(logits); free(probs); return 0;
}


In [ ]:
# Auto-grade Exercise 1
import numpy as np, torch, torch.nn.functional as F, subprocess
torch.manual_seed(0); B, T, V, Vp = 2, 3, 5, 8
logits = torch.randn(B, T, Vp) * 5.0
logits.numpy().astype(np.float32).tofile("course/ch07_build/logits.bin")
subprocess.run(["gcc","-O3","-Wall","-fopenmp","-o","course/ch07_build/exercise1","course/ch07_build/exercise1.c","-lm"], check=True)
subprocess.run(["./course/ch07_build/exercise1", str(B),str(T),str(V),str(Vp)], check=True)
probs_c = np.fromfile("course/ch07_build/probs_ex1.bin", dtype=np.float32).reshape(B, T, Vp)
probs_pt = F.softmax(logits[:, :, :V], dim=-1).numpy()
err_real = np.max(np.abs(probs_c[:, :, :V] - probs_pt))
pad_zero = np.all(probs_c[:, :, V:] == 0.0)
sums = probs_c[:, :, :V].sum(axis=-1)
print(f"real-V diff:       {err_real:.2e}")
print(f"row sums to ~1.0:  max deviation = {np.max(np.abs(sums-1.0)):.2e}")
print(f"padded slots zero: {pad_zero}")
print("PASS" if err_real < 1e-5 and pad_zero and np.max(np.abs(sums-1.0)) < 1e-5 else "FAIL")


### Solution to Exercise 1

In [ ]:
%%writefile course/ch07_build/exercise1_sol.c
#include <stdio.h>
#include <stdlib.h>
#include <math.h>
#include <omp.h>

void softmax_forward(float* probs, float* logits, int B, int T, int V, int Vp) {
    #pragma omp parallel for collapse(2)
    for (int b = 0; b < B; b++)
        for (int t = 0; t < T; t++) {
            float* logits_bt = logits + b*T*Vp + t*Vp;
            float* probs_bt  = probs  + b*T*Vp + t*Vp;
            float maxval = -10000.0f;
            for (int i = 0; i < V; i++) if (logits_bt[i] > maxval) maxval = logits_bt[i];
            float sum = 0.0f;
            for (int i = 0; i < V; i++) { probs_bt[i] = expf(logits_bt[i] - maxval); sum += probs_bt[i]; }
            for (int i = 0; i < V; i++) probs_bt[i] /= sum;
            for (int i = V; i < Vp; i++) probs_bt[i] = 0.0f;
        }
}

static void* rd(const char* p, size_t n){FILE*f=fopen(p,"rb");void*b=malloc(n);size_t r=fread(b,1,n,f);(void)r;fclose(f);return b;}

int main(int argc, char** argv) {
    int B=atoi(argv[1]), T=atoi(argv[2]), V=atoi(argv[3]), Vp=atoi(argv[4]);
    float* logits = (float*) rd("course/ch07_build/logits.bin", (size_t)B*T*Vp*sizeof(float));
    float* probs  = (float*) calloc((size_t)B*T*Vp, sizeof(float));
    softmax_forward(probs, logits, B, T, V, Vp);
    FILE* f=fopen("course/ch07_build/probs_ex1.bin","wb"); fwrite(probs,4,(size_t)B*T*Vp,f); fclose(f);
    free(logits); free(probs); return 0;
}


In [ ]:
!gcc -O3 -Wall -fopenmp -o course/ch07_build/exercise1_sol course/ch07_build/exercise1_sol.c -lm && ./course/ch07_build/exercise1_sol 2 3 5 8 && echo ran


## 14. TODO Exercise 2 — Write the Fused Backward

Fill in **the one inner-loop line** that materializes `(probs - indicator) * dloss`.


In [ ]:
%%writefile course/ch07_build/exercise2.c
#include <stdio.h>
#include <stdlib.h>

void crossentropy_softmax_backward(float* dlogits, float* dlosses, float* probs, int* targets,
                                   int B, int T, int V, int Vp) {
    for (int b = 0; b < B; b++)
        for (int t = 0; t < T; t++) {
            float* dlogits_bt = dlogits + b*T*Vp + t*Vp;
            float* probs_bt   = probs   + b*T*Vp + t*Vp;
            float dloss = dlosses[b*T + t];
            int ix = targets[b*T + t];
            for (int i = 0; i < V; i++) {
                float p = probs_bt[i];
                // TODO: indicator = 1 if i == ix else 0
                float indicator = 0.0f;
                indicator = (i == ix) ? 1.0f : 0.0f;
                // TODO: dlogits_bt[i] += (p - indicator) * dloss
                dlogits_bt[i] += (p - indicator) * dloss;
            }
            // (positions [V, Vp) intentionally untouched, stay at 0)
        }
}

static void* rd(const char* p, size_t n){FILE*f=fopen(p,"rb");void*b=malloc(n);size_t r=fread(b,1,n,f);(void)r;fclose(f);return b;}

int main(int argc, char** argv) {
    int B=atoi(argv[1]), T=atoi(argv[2]), V=atoi(argv[3]), Vp=atoi(argv[4]);
    float* probs   = (float*) rd("course/ch07_build/probs.bin",   (size_t)B*T*Vp*sizeof(float));
    float* dlosses = (float*) rd("course/ch07_build/dlosses.bin", (size_t)B*T*sizeof(float));
    int*   targets = (int*)   rd("course/ch07_build/targets.bin", (size_t)B*T*sizeof(int));
    float* dlogits = (float*) calloc((size_t)B*T*Vp, sizeof(float));
    crossentropy_softmax_backward(dlogits, dlosses, probs, targets, B, T, V, Vp);
    FILE* f=fopen("course/ch07_build/dlogits_ex2.bin","wb"); fwrite(dlogits,4,(size_t)B*T*Vp,f); fclose(f);
    free(probs); free(dlosses); free(targets); free(dlogits); return 0;
}


In [ ]:
# Auto-grade Exercise 2
import numpy as np, torch, torch.nn.functional as F, subprocess
torch.manual_seed(0); B, T, V, Vp = 2, 3, 5, 8
logits = (torch.randn(B, T, Vp) * 5.0).requires_grad_()
targets = torch.randint(0, V, (B, T))
loss = F.cross_entropy(logits[:, :, :V].reshape(B*T, V), targets.view(B*T), reduction='mean')
loss.backward()
probs_pt = F.softmax(logits.detach()[:, :, :V], dim=-1)
probs_full = torch.zeros(B, T, Vp); probs_full[:, :, :V] = probs_pt
probs_full.numpy().astype(np.float32).tofile("course/ch07_build/probs.bin")
targets.numpy().astype(np.int32).tofile("course/ch07_build/targets.bin")
torch.full((B, T), 1.0/(B*T), dtype=torch.float32).numpy().tofile("course/ch07_build/dlosses.bin")
subprocess.run(["gcc","-O3","-Wall","-o","course/ch07_build/exercise2","course/ch07_build/exercise2.c"], check=True)
subprocess.run(["./course/ch07_build/exercise2", str(B),str(T),str(V),str(Vp)], check=True)
dlogits_c = np.fromfile("course/ch07_build/dlogits_ex2.bin", dtype=np.float32).reshape(B,T,Vp)
err = np.max(np.abs(dlogits_c[:, :, :V] - logits.grad[:, :, :V].numpy()))
print(f"dlogits diff (real V): {err:.2e}")
print("PASS" if err < 1e-6 else "FAIL — check the (p - indicator) * dloss line")


### Solution to Exercise 2

In [ ]:
%%writefile course/ch07_build/exercise2_sol.c
#include <stdio.h>
#include <stdlib.h>

void crossentropy_softmax_backward(float* dlogits, float* dlosses, float* probs, int* targets,
                                   int B, int T, int V, int Vp) {
    for (int b = 0; b < B; b++)
        for (int t = 0; t < T; t++) {
            float* dlogits_bt = dlogits + b*T*Vp + t*Vp;
            float* probs_bt   = probs   + b*T*Vp + t*Vp;
            float dloss = dlosses[b*T + t];
            int ix = targets[b*T + t];
            for (int i = 0; i < V; i++) {
                float p = probs_bt[i];
                float indicator = (i == ix) ? 1.0f : 0.0f;
                dlogits_bt[i] += (p - indicator) * dloss;
            }
        }
}

static void* rd(const char* p, size_t n){FILE*f=fopen(p,"rb");void*b=malloc(n);size_t r=fread(b,1,n,f);(void)r;fclose(f);return b;}

int main(int argc, char** argv) {
    int B=atoi(argv[1]), T=atoi(argv[2]), V=atoi(argv[3]), Vp=atoi(argv[4]);
    float* probs   = (float*) rd("course/ch07_build/probs.bin",   (size_t)B*T*Vp*sizeof(float));
    float* dlosses = (float*) rd("course/ch07_build/dlosses.bin", (size_t)B*T*sizeof(float));
    int*   targets = (int*)   rd("course/ch07_build/targets.bin", (size_t)B*T*sizeof(int));
    float* dlogits = (float*) calloc((size_t)B*T*Vp, sizeof(float));
    crossentropy_softmax_backward(dlogits, dlosses, probs, targets, B, T, V, Vp);
    FILE* f=fopen("course/ch07_build/dlogits_ex2.bin","wb"); fwrite(dlogits,4,(size_t)B*T*Vp,f); fclose(f);
    free(probs); free(dlosses); free(targets); free(dlogits); return 0;
}


In [ ]:
!gcc -O3 -Wall -o course/ch07_build/exercise2_sol course/ch07_build/exercise2_sol.c && ./course/ch07_build/exercise2_sol 2 3 5 8 && echo ran


## Recap

You now know:

- **Softmax with the max-shift trick** is a 3-pass algorithm: max → exp+sum → normalize. Same as inside attention; this is the second time you've seen it.
- **Cross-entropy on a target index** is a 3-line function: gather, log, negate.
- The `Vp` (padded) vs `V` (real) distinction is everywhere in `llm.c`. Linear layers use `Vp` for matmul-friendliness; loss / softmax / their backwards loop only over `V` and zero out the rest.
- The fused softmax+CE backward is **`(probs - one_hot) * dloss`** — a 3-term inner loop with no division, no Jacobian matrix, no recomputation. This is one of the prettiest identities in deep learning, and the *only* reason loss heads are practical at vocab sizes of 50k+.
- "Fusion" — the practice of collapsing two consecutive ops into one — saves memory, FLOPs, and stability simultaneously. It's the core idea behind `fused_classifier.cuh` (Chapter 15) and several other CUDA kernels we'll meet later.

### What's next

**Chapter 8 — Putting it Together: Forward, Backward, AdamW.** This is the milestone chapter for Part I. You'll see how `llm.c` allocates **all** its parameters and activations as **one giant `malloc`** and points struct fields into the right offsets, how `gpt2_forward` orchestrates every layer you've now written, how `gpt2_backward` walks the layer chain in reverse, and how `gpt2_update` does AdamW with the m/v moment buffers. We'll end by running the repo's own `test_gpt2` binary to confirm everything we've learned reproduces real GPT-2 training step by step.
